In [46]:
# import os
# os.environ["SCIRPY_DATA_DIR"] = "/ix1/ylee/Yifan_Zhang/Code_data/external/VDJdb"
import scirpy as ir

In [47]:
import warnings
warnings.filterwarnings("ignore")

In [48]:
airr = ir.datasets.vdjdb()

In [49]:
ir.pp.index_chains(airr)
ir.tl.chain_qc(airr)

In [87]:
# ir.pp.ir_dist(airr, sequence='aa')
# ir.tl.define_clonotypes(airr, receptor_arms="all", 
#                         dual_ir="primary_only", 
#                         within_group="meta.subject.id",
#                         same_v_gene=True,
#                         same_j_gene=True)
# airr

In [50]:
# airr = airr[airr.obs['chain_pairing'].isin(['single pair'])]

In [51]:
airr

AnnData object with n_obs × n_vars = 140652 × 0
    obs: 'antigen.epitope', 'antigen.gene', 'antigen.species', 'meta.cell.subset', 'meta.clone.id', 'meta.donor.MHC', 'meta.donor.MHC.method', 'meta.epitope.id', 'meta.replica.id', 'meta.structure.id', 'meta.study.id', 'meta.subject.cohort', 'meta.subject.id', 'meta.tissue', 'method.frequency', 'method.identification', 'method.sequencing', 'method.singlecell', 'method.verification', 'mhc.a', 'mhc.b', 'mhc.class', 'reference.id', 'species', 'receptor_type', 'receptor_subtype', 'chain_pairing'
    uns: 'DB', 'chain_indices', 'scirpy_version'
    obsm: 'airr', 'chain_indices'

In [52]:
airr.obsm['airr'].fields

['consensus_count',
 'd_call',
 'd_cigar',
 'germline_alignment',
 'j_call',
 'j_cigar',
 'junction',
 'junction_aa',
 'locus',
 'productive',
 'rev_comp',
 'sequence',
 'sequence_alignment',
 'sequence_id',
 'v_call',
 'v_cigar']

In [53]:
meta_airr = ir.get.airr(airr, ["junction_aa", "v_call", "j_call"] ,  ('VJ_1', 'VDJ_1'))
airr.obs = airr.obs.join(meta_airr)

In [54]:
airr

AnnData object with n_obs × n_vars = 140652 × 0
    obs: 'antigen.epitope', 'antigen.gene', 'antigen.species', 'meta.cell.subset', 'meta.clone.id', 'meta.donor.MHC', 'meta.donor.MHC.method', 'meta.epitope.id', 'meta.replica.id', 'meta.structure.id', 'meta.study.id', 'meta.subject.cohort', 'meta.subject.id', 'meta.tissue', 'method.frequency', 'method.identification', 'method.sequencing', 'method.singlecell', 'method.verification', 'mhc.a', 'mhc.b', 'mhc.class', 'reference.id', 'species', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'VJ_1_junction_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_junction_aa', 'VDJ_1_v_call', 'VDJ_1_j_call'
    uns: 'DB', 'chain_indices', 'scirpy_version'
    obsm: 'airr', 'chain_indices'

In [56]:
import pandas as pd
import numpy as np

def junction_to_cdr3(s):
    """IMGT junction_aa -> CDR3: drop conserved leading Cys and trailing Phe/Trp.
    Returns object dtype (np.nan for missing) so AnnData writes it cleanly."""
    s = s.astype("string")
    s = s.str.replace(r"^C", "", regex=True)
    s = s.str.replace(r"[FW]$", "", regex=True)
    return s.astype(object).where(s.notna(), np.nan)

for chain in ["VJ_1", "VDJ_1"]:
    airr.obs[f"{chain}_cdr3_aa"] = junction_to_cdr3(airr.obs[f"{chain}_junction_aa"])

In [57]:
set(airr.obs['antigen.species'].tolist())

{'AdV',
 'AspergillusOryzae',
 'CMV',
 'CoxsackievirusB',
 'CryptococcusNeoforman',
 'CryptomeriaJaponica',
 'DENV',
 'E.Coli',
 'EBV',
 'FusariumOxysporum',
 'GallusGallus',
 'HCV',
 'HCoV-HKU1',
 'HHV',
 'HIV',
 'HIV-1',
 'HPV',
 'HPV-16',
 'HSV-2',
 'HTLV-1',
 'Homo Sapiens',
 'HomoSapiens',
 'InfluenzaA',
 'InfluenzaB',
 'KlebsiellaOxytoca',
 'LCMV',
 'M.tuberculosis',
 'MCMV',
 'MCPyV',
 'ManducaSexta',
 'MusMusculus',
 'PlasmodiumBerghei',
 'PlasmodiumFalciparum',
 'PseudomonasAeruginosa',
 'PseudomonasFluorescens',
 'RSV',
 'RotavirusA',
 'SARS-CoV',
 'SARS-CoV-2',
 'SIV',
 'SaccharomycesCerevisiae',
 'StreptomycesKanamyceticus',
 'Synthetic',
 'TriticumAestivum',
 'Trypanosoma cruzi',
 'VSV',
 'VZV',
 'Wheat',
 'YFV',
 'synthetic'}

In [58]:
SPECIES_GROUP_MAP = {
    # Viruses
    "CMV":              "Virus",
    "CoxsackievirusB":  "Virus",
    "DENV":             "Virus",
    "EBV":              "Virus",
    "HCV":              "Virus",
    "HCoV-HKU1":        "Virus",
    "HHV":              "Virus",
    "HIV":              "Virus",
    "HIV-1":            "Virus",
    "HPV":              "Virus",
    "HPV-16":           "Virus",
    "HSV-2":            "Virus",
    "HTLV-1":           "Virus",
    "InfluenzaA":       "Virus",
    "InfluenzaB":       "Virus",
    "LCMV":             "Virus",
    "MCMV":             "Virus",
    "MCPyV":            "Virus",
    "RotavirusA":       "Virus",
    "SARS-CoV":         "Virus",
    "SARS-CoV-2":       "Virus",
    "VSV":              "Virus",
    "VZV":              "Virus",
    "YFV":              "Virus",
    # Bacteria & parasites
    "E.Coli":                       "Bacteria & Parasite",
    "KlebsiellaOxytoca":            "Bacteria & Parasite",
    "M.tuberculosis":               "Bacteria & Parasite",
    "PlasmodiumBerghei":            "Bacteria & Parasite",
    "PlasmodiumFalciparum":         "Bacteria & Parasite",
    "PseudomonasAeruginosa":        "Bacteria & Parasite",
    "PseudomonasFluorescens":       "Bacteria & Parasite",
    "StreptomycesKanamyceticus":    "Bacteria & Parasite",
    "Trypanosoma cruzi":            "Bacteria & Parasite",
    # Fungi & plants
    "AspergillusOryzae":        "Fungi & Plant",
    "CryptococcusNeoforman":    "Fungi & Plant",
    "CryptomeriaJaponica":      "Fungi & Plant",
    "FusariumOxysporum":        "Fungi & Plant",
    "SaccharomycesCerevisiae":  "Fungi & Plant",
    "TriticumAestivum":         "Fungi & Plant",
    "Wheat":                    "Fungi & Plant",
    # Host organisms
    "GallusGallus":  "Host Organism",
    "Homo Sapiens":  "Host Organism",
    "HomoSapiens":   "Host Organism",
    "ManducaSexta":  "Host Organism",
    "MusMusculus":   "Host Organism",
    # Synthetic
    "Synthetic": "Synthetic/Unknown",
    "synthetic": "Synthetic/Unknown",
}

airr.obs["antigen.species_group"] = (
    airr.obs["antigen.species"]
    .map(SPECIES_GROUP_MAP)
    .fillna("Synthetic/Unknown")
    .astype("category")
)

print(airr.obs["antigen.species_group"].value_counts())

antigen.species_group
Virus                  85187
Host Organism          51978
Synthetic/Unknown       1959
Bacteria & Parasite     1259
Fungi & Plant            269
Name: count, dtype: int64


In [106]:
airr

AnnData object with n_obs × n_vars = 140652 × 0
    obs: 'antigen.epitope', 'antigen.gene', 'antigen.species', 'meta.cell.subset', 'meta.clone.id', 'meta.donor.MHC', 'meta.donor.MHC.method', 'meta.epitope.id', 'meta.replica.id', 'meta.structure.id', 'meta.study.id', 'meta.subject.cohort', 'meta.subject.id', 'meta.tissue', 'method.frequency', 'method.identification', 'method.sequencing', 'method.singlecell', 'method.verification', 'mhc.a', 'mhc.b', 'mhc.class', 'reference.id', 'species', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'VJ_1_junction_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_junction_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'VJ_1_cdr3_aa', 'VDJ_1_cdr3_aa', 'antigen.species_group', 'tcrdist_threshold', 'auc'
    uns: 'DB', 'chain_indices', 'scirpy_version', 'ir_dist_aa_identity', 'ir_dist_nt_identity'
    obsm: 'airr', 'chain_indices'

In [107]:
# for c in airr.obs.select_dtypes(include="string").columns:
#     mask = airr.obs[c].notna()
#     airr.obs[c] = airr.obs[c].astype(object).where(mask, np.nan)
airr.write("/ix1/ylee/Yifan_Zhang/Code_data/external/VDJdb/vdjdb_processed.h5ad")

### Filter

In [104]:
vc = airr.obs['meta.epitope.id'].value_counts()
(vc > 2).sum()

np.int64(167)

In [60]:
obs = airr.obs.copy()

# Step 1: Human only
human = obs[obs['species'] == 'HomoSapiens']
print(f"Human records: {len(human)}")  # typically ~80-90k

# Step 2: Viral antigens only
VIRAL = ['CMV', 'EBV', 'InfluenzaA', 'SARS-CoV-2', 'HSV', 'HCV', 'HBV']
viral = human[human['antigen.species'].isin(VIRAL)]
print(f"Human + viral: {len(viral)}")  # typically ~30-50k


Human records: 127045
Human + viral: 76596


In [61]:
viral_airr = airr[airr.obs['antigen.species_group'].isin(['Virus'])]
# ir.pp.index_chains(viral_airr)
# ir.tl.chain_qc(viral_airr)
viral_airr

View of AnnData object with n_obs × n_vars = 85187 × 0
    obs: 'antigen.epitope', 'antigen.gene', 'antigen.species', 'meta.cell.subset', 'meta.clone.id', 'meta.donor.MHC', 'meta.donor.MHC.method', 'meta.epitope.id', 'meta.replica.id', 'meta.structure.id', 'meta.study.id', 'meta.subject.cohort', 'meta.subject.id', 'meta.tissue', 'method.frequency', 'method.identification', 'method.sequencing', 'method.singlecell', 'method.verification', 'mhc.a', 'mhc.b', 'mhc.class', 'reference.id', 'species', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'VJ_1_junction_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_junction_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'VJ_1_cdr3_aa', 'VDJ_1_cdr3_aa', 'antigen.species_group'
    uns: 'DB', 'chain_indices', 'scirpy_version'
    obsm: 'airr', 'chain_indices'

## TCR dist

In [62]:
"""
TCRdist workflow using VDJdb data from scirpy.
- Subsets airr by k random epitope categories (meta.epitope.id)
- For each subset: largest clone = reference, rest = query → TCRrep
- Merges all subsets and repeats the same analysis

Chain: alpha-beta (paired)
"""

import random
import pandas as pd
import numpy as np
import scirpy as ir
from tcrdist.repertoire import TCRrep

# ── CONFIG ────────────────────────────────────────────────────────────────────
K           = 5       # <-- number of random epitope categories to sample
RANDOM_SEED = 42
MIN_ROWS    = 2       # minimum valid paired-chain rows to attempt TCRrep
# ─────────────────────────────────────────────────────────────────────────────

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [63]:
# # ── 2. Sample k epitope categories ───────────────────────────────────────────
# all_epitopes = airr.obs["meta.epitope.id"].dropna().unique().tolist()
# if K > len(all_epitopes):
#     raise ValueError(f"K={K} exceeds available epitope categories ({len(all_epitopes)})")

In [64]:
# ── 2. Sample in Virus  ───────────────────────────────────────────
all_epitopes = viral_airr.obs["meta.epitope.id"].dropna().unique().tolist()
if K > len(all_epitopes):
    raise ValueError(f"K={K} exceeds available epitope categories ({len(all_epitopes)})")

sampled_epitopes = random.sample(all_epitopes, K)
print(f"\nSampled {K} epitopes: {sampled_epitopes}")


Sampled 5 epitopes: ['13701', 'M1', 'KF11', 'GLC', 'pp65']


### dist to largest clone

In [65]:

# # sampled_epitopes = random.sample(all_epitopes, K)
# # print(f"\nSampled {K} epitopes: {sampled_epitopes}")

# # ── 3. V/J gene name normalisation ───────────────────────────────────────────
# # VDJdb stores genes as e.g. "TRAV1-2", "TRBV20-1".
# # tcrdist3 expects IMGT names with allele suffix: "TRAV1-2*01".
# # Rows whose V gene is not in the tcrdist3 reference DB are silently dropped
# # to an empty clone_df, causing the downstream pmhc_a_aa KeyError.


# def normalize_gene(name: str) -> str:
#     """Append *01 allele if no allele is present."""
#     if pd.isna(name):
#         return name
#     name = str(name).strip()
#     if "*" not in name:
#         name = name + "*01"
#     return name

# def build_tcrdist_df(obs_df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Map scirpy/VDJdb obs columns → TCRrep column names for paired alpha-beta.
#     Normalises V/J gene names and drops rows missing required fields.
#     Only the 6 core columns are included so that extra NaN-bearing columns
#     do not silently remove rows during TCRrep's deduplication step.
#     """
#     col_map = {
#         # Alpha chain
#         "VJ_1_v_call":   "v_a_gene",
#         "VJ_1_j_call":   "j_a_gene",
#         "VJ_1_junction_aa": "cdr3_a_aa",
#         # Beta chain
#         "VDJ_1_v_call":  "v_b_gene",
#         "VDJ_1_j_call":  "j_b_gene",
#         "VDJ_1_junction_aa": "cdr3_b_aa",
#     }
#     df = obs_df.rename(columns={k: v for k, v in col_map.items() if k in obs_df.columns})

#     required = ["v_a_gene", "j_a_gene", "cdr3_a_aa", "v_b_gene", "j_b_gene", "cdr3_b_aa"]
#     missing = [c for c in required if c not in df.columns]
#     if missing:
#         raise KeyError(f"Missing required columns: {missing}")

#     # Keep only the 6 core columns (extra columns with NaN cause silent row drops)
#     df = df[required].copy()

#     # Normalise gene names
#     for g in ["v_a_gene", "j_a_gene", "v_b_gene", "j_b_gene"]:
#         df[g] = df[g].apply(normalize_gene)

#     df = df.dropna().reset_index(drop=True)
#     df["count"] = 1   # required by TCRrep
#     return df


# # ── 4. Largest-clone selection ────────────────────────────────────────────────
# def get_largest_clone_idx(df: pd.DataFrame) -> int:
#     """
#     Return the integer-position index of the most frequent CDR3β sequence
#     (proxy for largest clone when explicit clone counts are absent).
#     """
#     freq    = df["cdr3_b_aa"].value_counts()
#     top_seq = freq.index[0]
#     # iloc position of first occurrence
#     return int(np.where(df["cdr3_b_aa"].values == top_seq)[0][0])


# # ── 5. Core TCRrep function ───────────────────────────────────────────────────
# def run_tcrrep(ref_df: pd.DataFrame,
#                query_df: pd.DataFrame,
#                label: str) -> dict:
#     """
#     Build a TCRrep on reference sequences, then compute rectangular
#     (reference × query) distances.

#     Returns a dict with:
#       tr        – the TCRrep object (pairwise distances on ref)
#       rw_alpha  – rectangular alpha distances (n_ref × n_query)
#       rw_beta   – rectangular beta  distances (n_ref × n_query)
#       rw_paired – sum of rw_alpha + rw_beta
#     """
    
#     print(f"\n  [{label}] reference={len(ref_df)} | query={len(query_df)}")

#     # ── 5a. Reference TCRrep (pairwise distances within reference) ────────────
#     tr = TCRrep(
#         cell_df=ref_df.copy(),
#         organism="human",
#         chains=["alpha", "beta"],
#         db_file="alphabeta_gammadelta_db.tsv",
#         compute_distances=True,
#     )

#     n_ref = len(tr.clone_df)
#     if n_ref == 0:
#         incomplete = tr.show_incomplete()
#         raise RuntimeError(
#             f"All reference rows dropped (unrecognised V genes). "
#             f"Incomplete rows:\n{incomplete}"
#         )
#     print(f"  [{label}] reference clone_df rows after V-gene filtering: {n_ref}")

#     # ── 5b. Query TCRrep (needed to get query clone_df) ──────────────────────
#     tr_q = TCRrep(
#         cell_df=query_df.copy(),
#         organism="human",
#         chains=["alpha", "beta"],
#         db_file="alphabeta_gammadelta_db.tsv",
#         compute_distances=False,   # skip pairwise; we only need clone_df
#     )

#     n_query = len(tr_q.clone_df)
#     if n_query == 0:
#         incomplete = tr_q.show_incomplete()
#         raise RuntimeError(
#             f"All query rows dropped (unrecognised V genes). "
#             f"Incomplete rows:\n{incomplete}"
#         )
#     print(f"  [{label}] query    clone_df rows after V-gene filtering: {n_query}")

#     # ── 5c. Rectangular distances (reference × query) ─────────────────────────
#     # compute_rect_distances(df=reference_clone_df, df2=query_clone_df)
#     # results are stored in tr.rw_alpha, tr.rw_beta
#     tr.compute_rect_distances(df=tr.clone_df, df2=tr_q.clone_df)

#     rw_alpha  = tr.rw_alpha                       # shape (n_ref, n_query)
#     rw_beta   = tr.rw_beta
#     rw_paired = rw_alpha + rw_beta

#     print(f"  [{label}] rw_alpha shape  : {rw_alpha.shape}")
#     print(f"  [{label}] rw_beta  shape  : {rw_beta.shape}")
#     print(f"  [{label}] mean paired dist: {rw_paired.mean():.1f}")

#     return dict(tr=tr, tr_q=tr_q,
#                 rw_alpha=rw_alpha, rw_beta=rw_beta, rw_paired=rw_paired)


# # ── 6. Per-epitope loop ───────────────────────────────────────────────────────
# results   = {}    # epitope → result dict
# subset_dfs = []   # accumulate obs for merged analysis

# for epitope in sampled_epitopes:
#     mask       = airr.obs["meta.epitope.id"] == epitope
#     subset_obs = airr.obs[mask].copy()
#     subset_dfs.append(subset_obs)

#     print(f"\n── Epitope: {epitope}  (n={mask.sum()}) ──")

#     try:
#         df = build_tcrdist_df(subset_obs)
#     except (KeyError, Exception) as e:
#         print(f"  Skipping {epitope}: {e}")
#         continue

#     if len(df) < MIN_ROWS:
#         print(f"  Skipping {epitope}: only {len(df)} valid paired-chain rows.")
#         continue

#     # Largest clone → reference (single row); remaining → query
#     ref_pos  = get_largest_clone_idx(df)
#     ref_df   = df.iloc[[ref_pos]].reset_index(drop=True)
#     query_df = df.drop(index=ref_pos).reset_index(drop=True)

#     print(f"  Reference CDR3β: {ref_df['cdr3_b_aa'].iloc[0]}")

#     try:
#         res = run_tcrrep(ref_df, query_df, label=epitope)
#         results[epitope] = res
#     except Exception as e:
#         print(f"  TCRrep failed for {epitope}: {e}")

# print(f"\n✓ Per-epitope complete: {len(results)}/{K} epitopes succeeded.")


# # ── 7. Merged analysis ────────────────────────────────────────────────────────
# print("\n══ Merged analysis (all sampled subsets combined) ══")

# merged_obs = pd.concat(subset_dfs, ignore_index=True)

# try:
#     merged_df = build_tcrdist_df(merged_obs)
# except Exception as e:
#     raise RuntimeError(f"Could not build merged TCR DataFrame: {e}")

# print(f"Merged valid paired-chain rows: {len(merged_df)}")

# if len(merged_df) < MIN_ROWS:
#     raise RuntimeError(f"Too few rows in merged set ({len(merged_df)}) to run TCRrep.")

# ref_pos_m  = get_largest_clone_idx(merged_df)
# ref_df_m   = merged_df.iloc[[ref_pos_m]].reset_index(drop=True)
# query_df_m = merged_df.drop(index=ref_pos_m).reset_index(drop=True)

# print(f"Merged reference CDR3β: {ref_df_m['cdr3_b_aa'].iloc[0]}")

# res_merged = run_tcrrep(ref_df_m, query_df_m, label="MERGED")


# # ── 8. Summary ────────────────────────────────────────────────────────────────
# print("\n══ Summary ══")
# header = f"{'Epitope':42s} {'ref_clones':>10} {'query_clones':>12} {'mean_paired_dist':>16}"
# print(header)
# print("─" * len(header))

# for ep, res in results.items():
#     n_ref   = res["rw_paired"].shape[0]
#     n_query = res["rw_paired"].shape[1]
#     mean_d  = res["rw_paired"].mean()
#     print(f"  {ep:40s} {n_ref:>10} {n_query:>12} {mean_d:>16.1f}")

# rw = res_merged["rw_paired"]
# print(f"  {'MERGED':40s} {rw.shape[0]:>10} {rw.shape[1]:>12} {rw.mean():>16.1f}")

# # -----------------------------------------------------------------------------
# # 10. KDE plot — K per-epitope distributions + MERGED
# # -----------------------------------------------------------------------------
# import matplotlib.pyplot as plt
# from scipy.stats import gaussian_kde
 
# fig, ax = plt.subplots(figsize=(9, 5))
 
# cmap          = plt.get_cmap("tab10")
# subset_items  = list(results.items())
# all_series    = subset_items + [("MERGED", res_merged)]
# colors        = [cmap(i) for i in range(len(subset_items))] + ["black"]
 
# for idx, (label, res) in enumerate(all_series):
#     distances = res["rw_paired"].flatten().astype(float)
#     distances = distances[np.isfinite(distances)]
 
#     if len(distances) < 2:
#         print(f"  Skipping KDE for {label}: fewer than 2 distances.")
#         continue
 
#     kde   = gaussian_kde(distances, bw_method="scott")
#     xs    = np.linspace(distances.min(), distances.max(), 500)
#     ys    = kde(xs)
#     color = colors[idx]
 
#     is_merged  = label == "MERGED"
#     lw         = 3.0 if is_merged else 2.0
#     ls         = "--" if is_merged else "-"
#     fill_alpha = 0.05 if is_merged else 0.10
#     short_lbl  = label if len(label) <= 35 else label[:33] + "..."
 
#     ax.plot(xs, ys, lw=lw, ls=ls, color=color, label=short_lbl,
#             zorder=3 if is_merged else 2)
#     ax.fill_between(xs, ys, alpha=fill_alpha, color=color)
 
# ax.set_xlabel("Paired TCRdist (alpha + beta chain)", fontsize=12)
# ax.set_ylabel("Density", fontsize=12)
# ax.set_title("TCRdist distribution: reference (largest clone) vs query TCRs", fontsize=13)
# ax.legend(title="Condition", fontsize=9, title_fontsize=9,
#           loc="upper right", framealpha=0.9)
# ax.spines[["top", "right"]].set_visible(False)
# plt.tight_layout()
 
# out_path = "tcrdist_kde.png"
# fig.savefig(out_path, dpi=150)
# print(f"\nKDE plot saved -> {out_path}")
# plt.show()

## Inner-outer epitope thresh

In [66]:
# ── 2. Sample in Virus  ───────────────────────────────────────────
K = 5
all_epitopes = viral_airr.obs["meta.epitope.id"].dropna().unique().tolist()
if K > len(all_epitopes):
    raise ValueError(f"K={K} exceeds available epitope categories ({len(all_epitopes)})")

sampled_epitopes = random.sample(all_epitopes, K)
print(f"\nSampled {K} epitopes: {sampled_epitopes}")


Sampled 5 epitopes: ['NP', 'M38', 'NS3-1073', 'GLC', 'NV9']


In [101]:
"""
TCRdist-based specificity threshold analysis.

Workflow
--------
1. Pull paired alpha/beta TCRs + known epitope from an AIRR AnnData (`airr`).
2. Compute pairwise TCRdist with tcrdist3's TCRrep.
3. For each antigen, split pairs into:
       inner-antigen    : both TCRs bind the same antigen
       external-antigen : exactly one TCR binds that antigen, the other binds something else
4. Find the tcrdist threshold that best separates same- vs different-specificity
   pairs (Youden's J on the ROC curve), globally and per antigen.

Assumptions
-----------
* `airr` is already loaded (AnnData, one row per cell in .obs).
* Known epitope label lives in airr.obs["meta.epitope.id"].
* Set ORGANISM correctly ("human" or "mouse") -- gene names must match the db.
"""

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc
from tcrdist.repertoire import TCRrep

# ----------------------------------------------------------------------
# config
# ----------------------------------------------------------------------
ORGANISM = "human"          # <-- "mouse" for mouse data; must match gene nomenclature
EPITOPE_COL = "antigen.epitope"
MAX_CLONES = 6000           # dense pairwise cap; raise/lower to fit memory
SEED = 0
MIN_CLONES_PER_EPITOPE = 5   # only analyze antigens with MORE than this many distinct clones; 0/None disables

col_map = {
    "VJ_1_v_call":       "v_a_gene",
    "VJ_1_j_call":       "j_a_gene",
    "VJ_1_junction_aa":  "cdr3_a_aa",
    "VDJ_1_v_call":      "v_b_gene",
    "VDJ_1_j_call":      "j_b_gene",
    "VDJ_1_junction_aa": "cdr3_b_aa",
}

# ----------------------------------------------------------------------
# 1. build the tcrdist3 input frame
# ----------------------------------------------------------------------
def _ensure_allele(g):
    """tcrdist3 wants IMGT names with an allele suffix, e.g. TRBV20-1*01."""
    g = str(g)
    return g if ("*" in g or g in ("nan", "")) else g + "*01"

# def filter_rare_epitopes(df, min_clones, clone_cols=None):
#     """Keep only epitopes with > min_clones distinct clonotypes.

#     A clone = a unique paired-chain TCR (the six V/J/CDR3 fields). Antigens
#     with too few clones give too few inner-antigen pairs for a stable ROC,
#     so drop them before the dense pairwise step.
#     """
#     if not min_clones or min_clones <= 0:
#         return df.reset_index(drop=True)
#     if clone_cols is None:
#         clone_cols = ["v_a_gene", "j_a_gene", "cdr3_a_aa",
#                       "v_b_gene", "j_b_gene", "cdr3_b_aa"]
#     n_clones = df.drop_duplicates(clone_cols).groupby("epitope").size()
#     keep = n_clones[n_clones > min_clones].index
#     out = df[df["epitope"].isin(keep)].reset_index(drop=True)
#     print(f"epitope filter (>{min_clones} clones): kept {len(keep)}/"
#           f"{df['epitope'].nunique()} antigens, {len(out)}/{len(df)} cells")
#     return out


def build_tcrdist_df(adata, col_map, epitope_col=EPITOPE_COL):
    cols = list(col_map.keys()) + [epitope_col]
    df = adata.obs[cols].rename(columns=col_map)
    df = df.rename(columns={epitope_col: "epitope"})

    needed = ["v_a_gene", "j_a_gene", "cdr3_a_aa",
              "v_b_gene", "j_b_gene", "cdr3_b_aa", "epitope"]
    df = df.replace("", np.nan).dropna(subset=needed).copy()

    # AnnData .obs string columns are pandas 'category' dtype. tcrdist3
    # deduplicates with groupby(index_cols)['count'].sum(); on categoricals
    # pandas (observed=False) reindexes to the FULL cartesian product of all
    # category levels, which overflows -> "Product space too large to allocate
    # arrays!". Cast to plain strings so only observed combinations are grouped.
    df[needed] = df[needed].astype(str)

    for c in ["v_a_gene", "j_a_gene", "v_b_gene", "j_b_gene"]:
        df[c] = df[c].map(_ensure_allele)

    df["count"] = 1
    return df.reset_index(drop=True)

def stratified_subsample(df, total_cap, seed=SEED):
    """Cap total clones for the dense N x N matrix while keeping every antigen.

    Roughly even budget per antigen (rare antigens kept whole), then trimmed
    to total_cap. Subsampling only thins the pairwise distributions; it does
    not bias the inner- vs external-antigen comparison.
    """
    if len(df) <= total_cap:
        return df.reset_index(drop=True)
    per = max(2, total_cap // df["epitope"].nunique())
    parts = [g.sample(min(len(g), per), random_state=seed)
             for _, g in df.groupby("epitope")]
    out = pd.concat(parts)
    if len(out) > total_cap:
        out = out.sample(total_cap, random_state=seed)
    return out.reset_index(drop=True)

# dist for subset
# airr_sub = airr[airr.obs["meta.epitope.id"].isin(sampled_epitopes)].copy()
# clone_df = build_tcrdist_df(airr_sub, col_map)

# dist for filtered high epitopes
# clone_df = build_tcrdist_df(airr, col_map)
# clone_df = filter_rare_epitopes(clone_df, MIN_CLONES_PER_EPITOPE)   # <-- new

# print(f"{len(clone_df)} paired cells with a known epitope "
#       f"across {clone_df['epitope'].nunique()} antigens")

# clone_df = stratified_subsample(clone_df, MAX_CLONES)

# dist for all
clone_df = build_tcrdist_df(airr, col_map)
print(f"{len(clone_df)} paired cells with a known epitope "
      f"across {clone_df['epitope'].nunique()} antigens")
clone_df = stratified_subsample(clone_df, MAX_CLONES)
print(f"-> using {len(clone_df)} clones after subsampling (cap {MAX_CLONES})")

# ----------------------------------------------------------------------
# 2. pairwise TCRdist
# ----------------------------------------------------------------------
tr = TCRrep(
    cell_df=clone_df,
    organism=ORGANISM,
    chains=["alpha", "beta"],
    db_file="alphabeta_gammadelta_db.tsv",
    compute_distances=True,
)

# combined paired-chain distance; tr.clone_df keeps the epitope column
D   = tr.pw_alpha + tr.pw_beta
epi = tr.clone_df["epitope"].to_numpy()
n   = D.shape[0]
# single-chain instead? set chains=["beta"] above and use D = tr.pw_beta

# ----------------------------------------------------------------------
# 3. + 4. threshold analysis
# ----------------------------------------------------------------------
iu = np.triu_indices(n, k=1)            # each unordered pair once, no self-pairs
d  = D[iu]
ei, ej = epi[iu[0]], epi[iu[1]]

def best_threshold(inner_d, external_d):
    """Youden-J optimal tcrdist cutoff; pairs with dist <= thr -> 'same antigen'."""
    if len(inner_d) == 0 or len(external_d) == 0:
        return np.nan, np.nan
    y = np.r_[np.ones(len(inner_d)), np.zeros(len(external_d))]
    s = -np.r_[inner_d, external_d]      # higher score = closer = more likely same
    fpr, tpr, thr = roc_curve(y, s)
    k = np.argmax(tpr - fpr)
    return -thr[k], auc(fpr, tpr)        # convert score cutoff back to a distance

# ---- global (same vs different antigen, all antigens pooled) ----
same = ei == ej
g_thr, g_auc = best_threshold(d[same], d[~same])
print(f"\nGLOBAL  threshold = {g_thr:.0f}   AUC = {g_auc:.3f}")
print(f"  same-antigen median dist = {np.median(d[same]):.0f}")
print(f"  diff-antigen median dist = {np.median(d[~same]):.0f}")

# ---- per antigen ----
rows = []
for ag in pd.unique(epi):
    a = ei == ag
    b = ej == ag
    inner    = d[a & b]          # both bind ag
    external = d[a ^ b]          # exactly one binds ag
    thr, roc_auc = best_threshold(inner, external)
    rows.append({
        "antigen": ag,
        "n_clones": int((epi == ag).sum()),
        "n_inner_pairs": len(inner),
        "n_external_pairs": len(external),
        "inner_median": np.median(inner) if len(inner) else np.nan,
        "external_median": np.median(external) if len(external) else np.nan,
        "tcrdist_threshold": thr,
        "auc": roc_auc,
    })

res = (pd.DataFrame(rows)
       .sort_values("auc", ascending=False, na_position="last")
       .reset_index(drop=True))
print("\nPer-antigen specificity thresholds:")
print(res.shape)

87684 paired cells with a known epitope across 1540 antigens
-> using 2903 clones after subsampling (cap 6000)

GLOBAL  threshold = 242   AUC = 0.595
  same-antigen median dist = 273
  diff-antigen median dist = 284

Per-antigen specificity thresholds:
(1484, 8)


In [102]:
res.head(5)

,antigen,n_clones,n_inner_pairs,n_external_pairs,inner_median,external_median,tcrdist_threshold,auc
0,HYPYRLWHY,3,3,8139,95.0,274.0,100.0,1.0
1,NCTFEYVSQPFLMDL,3,3,8139,82.0,308.0,94.0,1.0
2,RTKDIKDVFY,3,3,8139,84.0,277.0,131.0,1.0
3,TAFTIPSI,3,3,8139,84.0,261.0,84.0,1.0
4,TLDYKPLSV,2,1,5428,69.0,286.0,69.0,1.0


In [80]:
res = pd.DataFrame(rows).set_index("antigen")

# per-antigen lookup -> broadcast onto every cell via its epitope id
airr.obs["tcrdist_threshold"] = airr.obs["meta.epitope.id"].map(res["tcrdist_threshold"])
airr.obs["auc"]               = airr.obs["meta.epitope.id"].map(res["auc"])

airr

AnnData object with n_obs × n_vars = 140652 × 0
    obs: 'antigen.epitope', 'antigen.gene', 'antigen.species', 'meta.cell.subset', 'meta.clone.id', 'meta.donor.MHC', 'meta.donor.MHC.method', 'meta.epitope.id', 'meta.replica.id', 'meta.structure.id', 'meta.study.id', 'meta.subject.cohort', 'meta.subject.id', 'meta.tissue', 'method.frequency', 'method.identification', 'method.sequencing', 'method.singlecell', 'method.verification', 'mhc.a', 'mhc.b', 'mhc.class', 'reference.id', 'species', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'VJ_1_junction_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_junction_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'VJ_1_cdr3_aa', 'VDJ_1_cdr3_aa', 'antigen.species_group', 'tcrdist_threshold', 'auc'
    uns: 'DB', 'chain_indices', 'scirpy_version'
    obsm: 'airr', 'chain_indices'

In [81]:
aa

NameError: name 'aa' is not defined

In [ ]:
def plot_antigens(antigens, bins=40, ncols=3, density=True):
    """One panel per antigen: inner-antigen vs external-antigen TCRdist,
    with the Youden-J threshold and AUC annotated."""
    import matplotlib.pyplot as plt
 
    antigens = [a for a in antigens if (epi == a).any()]
    missing = [a for a in antigens if not (epi == a).any()]
    if missing:
        print(f"skipping antigens not present after subsampling: {missing}")
    if not antigens:
        raise ValueError("none of the requested antigens are present in epi")
 
    n = len(antigens)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(4.5 * ncols, 3.2 * nrows),
                             squeeze=False)
 
    for ax, ag in zip(axes.flat, antigens):
        a, b = ei == ag, ej == ag
        inner, external = d[a & b], d[a ^ b]
        thr, roc_auc = best_threshold(inner, external)
 
        if len(inner):
            ax.hist(inner, bins=bins, density=density, alpha=0.65,
                    color="#2c7fb8", label=f"inner (n={len(inner)})")
        if len(external):
            ax.hist(external, bins=bins, density=density, alpha=0.5,
                    color="#d95f0e", label=f"external (n={len(external)})")
        if not np.isnan(thr):
            ax.axvline(thr, color="k", ls="--", lw=1.2, label=f"thr={thr:.0f}")
 
        title = str(ag) + (f"  (AUC={roc_auc:.2f})" if not np.isnan(roc_auc) else "")
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("TCRdist")
        ax.set_ylabel("density" if density else "count")
        ax.legend(fontsize=8)
 
    for ax in axes.flat[n:]:        # hide unused panels
        ax.set_visible(False)
    fig.tight_layout()
    return fig
 
fig = plot_antigens(sampled_epitopes)
out_path = "antigen_repertoire_dist.png"
fig.savefig(out_path, dpi=150)